<a href="https://colab.research.google.com/github/VanamaAkhila/Speech_Emotion_detection_and_motivation_Generatin/blob/main/Speech_emotion_detection_and_Motivation_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Hugging Face

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q git+https://github.com/openai/whisper.git transformers accelerate sentencepiece edge-tts pygame scipy soundfile

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from google.colab import files
print("Upload a 5–10 second WAV file for speech input:")
audio_file = files.upload()
AUDIO_PATH = list(audio_file.keys())[0]
print("Uploaded:", AUDIO_PATH)

Upload a 5–10 second WAV file for speech input:


Saving Recording.m4a to Recording.m4a
Uploaded: Recording.m4a


In [ ]:
from huggingface_hub import login
login()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_id = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

llama_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=120,
    temperature=0.7,
    top_p=0.9
)

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_id = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=120,
    temperature=0.7,
    top_p=0.9
)

def generate_motivation(text: str, emotion: str) -> str:
    prompt = f"""You are a supportive and empathetic assistant.

User emotion: {emotion.upper()}
User said: "{text}"

Instructions:
- Write exactly 2-3 short motivational sentences.
- Be encouraging and positive.
- Do not give medical advice.
- Output only the motivational message.

Motivational message:"""

    response = generator(prompt)[0]["generated_text"]
    return response.split("Motivational message:")[-1].strip()

Device set to use cuda:0


In [ ]:
import whisper
from transformers import pipeline

print("Loading Whisper...")
whisper_model = whisper.load_model("base").cuda()

print("Loading Emotion Model...")
emotion_model = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,
    device=0
)

Loading Whisper...


100%|███████████████████████████████████████| 139M/139M [00:02<00:00, 58.0MiB/s]


Loading Emotion Model...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
result = whisper_model.transcribe(AUDIO_PATH, language="en")
text = result["text"].strip()

print("Transcribed Text:", text)

preds = emotion_model(text)[0]
emotion = max(preds, key=lambda x: x["score"])
print("Detected Emotion:", emotion["label"])

Transcribed Text: Hey, I got a job.
Detected Emotion: joy


In [ ]:
motivation = generate_motivation(text, emotion["label"])
print(motivation)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


"That's amazing news! I'm so happy for you. You're taking a huge step towards your goals. Keep pushing forward and know that you're capable of achieving anything you set your mind to."


In [ ]:
# Available voices (run list_voices() to see all):
#             - en-US-AriaNeural: Female, expressive and warm
#             - en-US-GuyNeural: Male, friendly and casual
#             - en-US-JennyNeural: Female, conversational
#             - en-US-ChristopherNeural: Male, professional
#             - en-GB-SoniaNeural: British Female, clear
#             - en-IN-NeerjaNeural: Indian Female
#             - en-IN-PrabhatNeural: Indian Male

import os, tempfile, edge_tts, asyncio
from IPython.display import Audio, display

output_path = os.path.join(tempfile.gettempdir(), "motivation.mp3")

async def speak():
    communicate = edge_tts.Communicate(text=motivation, voice="en-GB-SoniaNeural", rate="+2%", pitch="+2Hz")
    await communicate.save(output_path)

# Jupyter-safe execution
await speak()

display(Audio(output_path))